### nxsut — WP1 screening: size the energy perimeter

Data-driven confirmation of the priority ladder (`UPDATE_PLAN.md` §WP1) **before**
any data work. Loads the exported **v3.0** baseline table directly (no generation
pipeline, no nxbase API needed) and runs three native-MARIO analyses:

1. **GHG footprint decomposition** of the major energy vectors into upstream
   contributors (`fa_ex`/`fc_ex` machinery, here computed on the target columns) —
   evidence of which flows drive energy-chain footprints.
2. **Import share** of total use per commodity (from `U` + `Yc`) → trade-update
   candidates.
3. **Multi-producer** structure per commodity (from supply shares `s`) → supply-mix
   candidates.

Output: one classification table `commodity → {priority, trade, supply_mix}` with the
supporting metrics, committed to **`support/wp1_perimeter.csv`**.

GWP basket: **AR6 GWP-100** (matches the nxbase governed basket). Run from the repo
root; set `user`/`year` in the first code cell.

In [1]:
import gc
import os

import mario
import numpy as np
import pandas as pd
import yaml

import warnings
warnings.filterwarnings('ignore')

with open('paths.yml') as f:
    paths = yaml.safe_load(f)
user = 'LR'   # change to your username
year = 2023   # base year to screen (v3.0 export with the full matrix set)
paths = paths[user]

flows = os.path.join(paths['export'], 'v3.0', str(year), 'flows')
db = mario.parse_from_txt(flows, table='SUT', mode='flows')
db.meta.source = 'EXIOBASE Hybrid 3.3.18'
regions = list(db.get_index('Region'))
commodities = list(db.get_index('Commodity'))

# commodity units from the exported units file (goods vs services vs waste-service)
units_raw = pd.read_csv(os.path.join(flows, 'units.txt'), header=0)
units_raw.columns = ['level', 'item', 'unit']
unit_of = dict(zip(units_raw.loc[units_raw.level == 'Commodity', 'item'],
                   units_raw.loc[units_raw.level == 'Commodity', 'unit']))
GOODS_UNITS = {'tonnes', 'kg'}   # physical goods (services=Meuro; secondary/waste='tonnes (service)')
print(f'regions={len(regions)} commodities={len(commodities)}')

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/nxsut/v3.0/2023/flows in matrix mode (txt/csv).


INFO Parser: state payload ready with 10 canonical blocks.


INFO Parser: txt state ready for SUT.


INFO Metadata: initialized.


regions=48 commodities=197


#### Parameters and curated perimeter sets

The **P1 carriers** and the **P2 transition-material shortlist** are curated (they encode
the plan's priority ladder); the metrics below *confirm and annotate* them. A separate
`review_candidate` flag surfaces any P3 **good** that the decomposition shows to matter,
so surprises are visible without mislabelling services/waste as materials.

In [2]:
# AR6 GWP-100 (matches the nxbase governed basket)
GHG_AR6 = {
    'Carbon dioxide, fossil (air - Emiss)': 1.0,
    'Carbon dioxide, biogenic (air - Emiss)': 0.0,
    'CH4 (air - Emiss)': 29.8,   # fossil methane, AR6-100
    'N2O (air - Emiss)': 273.0,
    'SF6 (air - Emiss)': 25200.0,
}

# screening thresholds
IMPORT_MIN = 0.05        # >=5% of use imported -> trade candidate
SUPPLIER_SHARE = 0.05    # activity counts if it holds >=5% of a commodity's domestic supply
CONTRIB_MIN = 0.03       # upstream good >=3% of a major energy vector -> review candidate
SECONDARY_MIN = 0.02     # >=2% of a commodity's supply from recycling -> virgin/recycled mix candidate

CARRIER_LABELS = {
    'Anthracite', 'Coking Coal', 'Other Bituminous Coal', 'Sub-Bituminous Coal',
    'Patent Fuel', 'Lignite/Brown Coal', 'BKB/Peat Briquettes', 'Peat',
    'Coke Oven Coke', 'Gas Coke', 'Coal Tar', 'Charcoal',
    'Crude petroleum and services related to crude oil extraction; excluding surveying',
    'Natural gas and services related to natural gas extraction; excluding surveying',
    'Natural Gas Liquids', 'Other Hydrocarbons',
    'Motor Gasoline', 'Aviation Gasoline', 'Gasoline Type Jet Fuel', 'Kerosene Type Jet Fuel',
    'Kerosene', 'Gas/Diesel Oil', 'Heavy Fuel Oil', 'Refinery Gas',
    'Liquefied Petroleum Gases (LPG)', 'Refinery Feedstocks', 'Ethane', 'Naphtha',
    'White Spirit & SBP', 'Lubricants', 'Bitumen', 'Paraffin Waxes', 'Petroleum Coke',
    'Non-specified Petroleum Products', 'Additives/Blending Components',
    'Coke oven gas', 'Blast Furnace Gas', 'Oxygen Steel Furnace Gas', 'Gas Works Gas', 'Biogas',
    'Biogasoline', 'Biodiesels', 'Other Liquid Biofuels',
    'Nuclear fuel', 'Electricity', 'Electricity need',
    'Steam and hot water supply services',
    'Steam reforming hydrogen', 'Steam reforming hydrogen with CCS',
    'Coal gasification hydrogen', 'Coal gasification hydrogen with CCS', 'Electrolysis hydrogen',
}

# transition-material shortlist (WP1 expected P2): goods, ores, and secondary re-processing
P2_SHORTLIST = {
    'Basic iron and steel and of ferro-alloys and first products thereof': 'worldsteel BOF/EAF',
    'Secondary steel for treatment; Re-processing of secondary steel into new steel': 'worldsteel BOF/EAF',
    'Iron ores': '',
    'Aluminium and aluminium products': 'IAI',
    'Secondary aluminium for treatment; Re-processing of secondary aluminium into new aluminium': 'IAI',
    'Aluminium ores and concentrates': '',
    'Copper products': 'ICSG',
    'Secondary copper for treatment; Re-processing of secondary copper into new copper': 'ICSG',
    'Copper ores and concentrates': '',
    'Cement; lime and plaster': 'GCCA/GNR clinker ratio',
    'Ash for treatment; Re-processing of ash into clinker': 'GCCA/GNR clinker ratio',
    'Glass and glass products': 'FEVE (EU)',
    'Secondary glass for treatment; Re-processing of secondary glass into new glass': 'FEVE (EU)',
    'Plastics; basic': 'PlasticsEurope/OECD',
    'Secondary plastic for treatment; Re-processing of secondary plastic into new plastic': 'PlasticsEurope/OECD',
}

# per-country supply-mix statistic available (WP4 table)
SUPPLY_MIX_SOURCE = {
    'Electricity': 'EMBER (done)', 'Electricity need': 'EMBER (done)',
    'Steam and hot water supply services': 'IEA balances',
    'Basic iron and steel and of ferro-alloys and first products thereof': 'worldsteel BOF/EAF',
    'Aluminium and aluminium products': 'IAI',
    'Plastics; basic': 'PlasticsEurope/OECD',
}
POOLED = {'Electricity', 'Electricity need'}   # already pooled (v3.0); gas = pooling candidate

# major energy vectors used as decomposition targets (avoid niche-chain noise)
MAJOR_CHAINS = [
    'Electricity need', 'Electricity',
    'Natural gas and services related to natural gas extraction; excluding surveying',
    'Crude petroleum and services related to crude oil extraction; excluding surveying',
    'Gas/Diesel Oil', 'Motor Gasoline', 'Heavy Fuel Oil',
    'Other Bituminous Coal', 'Coking Coal', 'Lignite/Brown Coal',
    'Steam and hot water supply services',
]

E = db.query('E')
sat = list(E.index.get_level_values(-1)) if isinstance(E.index, pd.MultiIndex) else list(E.index)
gwp = {k: v for k, v in GHG_AR6.items() if k in set(sat)}
print('GHG satellite rows:', [s for s in sat if 'Emiss' in s])
print('GWP basket:', gwp)

INFO Resolver: resolving E for baseline.


INFO Resolver: trying E via concat.


INFO Resolver: resolved E via concat.


GHG satellite rows: ['Carbon dioxide, fossil (air - Emiss)', 'N2O (air - Emiss)', 'CH4 (air - Emiss)', 'HFCs (air - Emiss)', 'PFCs (air - Emiss)', 'SF6 (air - Emiss)', 'NOX  (air - Emiss)', 'SOx (air - Emiss)', 'NH3 (air - Emiss)', 'NMVOC (air - Emiss)', 'CO  (air - Emiss)', 'CFCs (air - Emiss)', 'HCFCs (air - Emiss)', 'Pb (air - Emiss)', 'Cd (air - Emiss)', 'Hg (air - Emiss)', 'As (air - Emiss)', 'Cr (air - Emiss)', 'Cu (air - Emiss)', 'Ni (air - Emiss)', 'Se (air - Emiss)', 'Zn (air - Emiss)', 'Aldrin (air - Emiss)', 'Chlordane (air - Emiss)', 'Chlordecone (air - Emiss)', 'Dieldrin (air - Emiss)', 'Endrin (air - Emiss)', 'Heptachlor (air - Emiss)', 'Hexabr.-biph. (air - Emiss)', 'Mirex (air - Emiss)', 'Toxaphene (air - Emiss)', 'HCH (air - Emiss)', 'DDT (air - Emiss)', 'PCB (air - Emiss)', 'dioxin (air - Emiss)', 'PM10 (air - Emiss)', 'PAH (total of 4 components, sum of EM_AIR.43, 45, 46, 47) (air - Emiss)', 'Benzene (air - Emiss)', '1,3 Butadiene (air - Emiss)', 'Formaldehyd (air - 

#### Step 0 — per-commodity GHG intensity and output

`fc` (commodity footprint) aggregated over the GHG basket gives the embodied GHG intensity
per unit of each commodity. Sanity: Italy's *Electricity need* intensity should land near
the known v3.0 value (~100 tCO2eq/TJ ≈ ~360 gCO2eq/kWh).

In [3]:
fc = db.query('fc')                                   # satellite x commodity
fc_ghg = sum(fc.loc[k] * v for k, v in gwp.items())   # GHG intensity per (region, commodity)
Xc = db.query('Xc')
Xc = Xc.iloc[:, 0] if isinstance(Xc, pd.DataFrame) else Xc

def by_commodity(series, weight):
    item = series.index.get_level_values('Item')
    w = weight.reindex(series.index).fillna(0.0)
    return (series * w).groupby(item).sum() / w.groupby(item).sum().replace(0, np.nan)

ghg_int = by_commodity(fc_ghg, Xc)   # output-weighted GHG intensity per commodity label
# EXIOBASE Hybrid satellite emissions are in tonnes -> footprints are in tCO2eq per commodity unit
sat_unit = db.units['Satellite account'].loc['Carbon dioxide, fossil (air - Emiss)', 'unit']
it_need = fc_ghg.loc[('IT', 'Commodity', 'Electricity need')]
print(f"satellite unit = {sat_unit!r} -> GHG footprint in {sat_unit}CO2eq per commodity unit")
print(f"SANITY IT 'Electricity need' = {it_need:.2f} tCO2eq/TJ (~{it_need*3.6:.0f} gCO2eq/kWh)")
del fc; gc.collect()

INFO Resolver: resolving fc for baseline.


INFO Resolver: trying fc via extract.


INFO Resolver: trying fc via formula build_sut_fc_from_ea_s_wcc (compute_method=inverse, runtime=inverse).


INFO Resolver: resolved wcc via formula build_sut_wcc_from_u_s (compute_method=inverse, runtime=inverse).


INFO Resolver: resolved fc via formula build_sut_fc_from_ea_s_wcc (compute_method=inverse, runtime=inverse).


INFO Resolver: resolving Xc for baseline.


INFO Resolver: trying Xc via formula build_sut_Xc_from_U_Yc.


INFO Resolver: resolved Xc via formula build_sut_Xc_from_U_Yc.


satellite unit = 'tonnes' -> GHG footprint in tonnesCO2eq per commodity unit
SANITY IT 'Electricity need' = 91.99 tCO2eq/TJ (~331 gCO2eq/kWh)


0

#### Step 1 — footprint decomposition of the major energy vectors

For each major energy vector *j*, the GHG footprint is decomposed into upstream
contributions: commodity-side `diag(ec_ghg)·wcc` and activity-side `diag(ea_ghg)·(s·wcc)`
(the exact terms behind MARIO's `fc_ex`), sliced on the target columns and output-weighted
across regions. Activity contributors are mapped to their main supplied commodity so the
ranking is by upstream **commodity**.

In [4]:
ea = db.query('ea'); ec = db.query('ec')
ea_ghg = sum(ea.loc[k] * v for k, v in gwp.items())
ec_ghg = sum(ec.loc[k] * v for k, v in gwp.items())
s = db.query('s')            # activity x commodity market shares
wcc = db.query('wcc')        # commodity x commodity Leontief

S = db.query('S')
def activity_main_commodity(S):
    out = {}
    for r in regions:
        blk = S.xs(r, level='Region', drop_level=False)
        cols_r = [c for c in S.columns if c[0] == r]
        am = blk[cols_r].values.argmax(axis=1)
        for i, act in enumerate(blk.index):
            out[act] = cols_r[am[i]][2]
    return pd.Series(out)
act_to_com = activity_main_commodity(S)
# virgin/recycled split: share of each commodity's domestic supply from a re-processing activity
RECYCLE_HINTS = ('Re-processing of secondary', 'Recycling of')
is_recycle = pd.Series([any(h in a for h in RECYCLE_HINTS)
                        for a in S.index.get_level_values('Item')], index=S.index)
col_item = S.columns.get_level_values('Item')
tot_by_com = S.sum(axis=0).groupby(col_item).sum()
rec_by_com = S.loc[is_recycle.values].sum(axis=0).groupby(col_item).sum()
secondary_share = (rec_by_com / tot_by_com.replace(0, np.nan)).reindex(commodities).fillna(0.0)
print('recycled supply share (materials):')
print(secondary_share[secondary_share >= SECONDARY_MIN].sort_values(ascending=False).round(3).to_string())
del S; gc.collect()

decomp = {}
for L in MAJOR_CHAINS:
    cols = [(r, 'Commodity', L) for r in regions if (r, 'Commodity', L) in wcc.columns]
    if not cols:
        continue
    wccJ = wcc[cols]
    comm_contrib = wccJ.mul(ec_ghg.reindex(wccJ.index).fillna(0.0), axis=0)
    act_contrib = s.dot(wccJ).mul(ea_ghg.reindex(s.index).fillna(0.0), axis=0)
    wcol = Xc.reindex(cols).fillna(0.0).values
    comm_s = pd.Series(comm_contrib.values @ wcol,
                       index=wccJ.index.get_level_values('Item')).groupby(level=0).sum()
    act_item = act_contrib.index.map(lambda a: act_to_com.get(a, a[2]))
    act_s = pd.Series(act_contrib.values @ wcol, index=act_item).groupby(level=0).sum()
    total = comm_s.add(act_s, fill_value=0.0)
    if total.sum() > 0:
        decomp[L] = (total / total.sum()).sort_values(ascending=False)

energychain_contrib = {}
for L, ser in decomp.items():
    for com, share in ser.items():
        if com != L:
            energychain_contrib[com] = max(energychain_contrib.get(com, 0.0), float(share))

for L in ['Electricity need',
          'Natural gas and services related to natural gas extraction; excluding surveying',
          'Gas/Diesel Oil']:
    if L in decomp:
        print(f"top upstream contributors to '{L}':")
        print(decomp[L].head(8).round(4).to_string(), '\n')
del wcc, s, comm_contrib, act_contrib; gc.collect()

INFO Resolver: resolving ea for baseline.


INFO Resolver: trying ea via extract.


INFO Resolver: trying ea via formula build_sut_ea_from_Ea_Xa.


INFO Resolver: resolved ea via formula build_sut_ea_from_Ea_Xa.


INFO Resolver: resolving ec for baseline.


...

INFO Resolver: trying wcc via formula build_sut_wcc_from_u_s (compute_method=inverse, runtime=inverse).


INFO Resolver: resolved wcc via formula build_sut_wcc_from_u_s (compute_method=inverse, runtime=inverse).


recycled supply share (materials):
Item
Secondary construction material for treatment; Re-processing of secondary construction material into aggregates                            1.000
Secondary other non-ferrous metals for treatment; Re-processing of secondary other non-ferrous metals into new other non-ferrous metals    1.000
Bottles for treatment; Recycling of bottles by direct reuse                                                                                1.000
Secondary steel for treatment; Re-processing of secondary steel into new steel                                                             1.000
Secondary raw materials                                                                                                                    1.000
Secondary preciuos metals for treatment; Re-processing of secondary preciuos metals into new preciuos metals                               1.000
Secondary plastic for treatment; Re-processing of secondary plastic into new plastic      

top upstream contributors to 'Electricity need':
Item
Electricity                                                                          0.8552
Other Bituminous Coal                                                                0.0636
Steam and hot water supply services                                                  0.0238
Natural gas and services related to natural gas extraction; excluding surveying      0.0234
Crude petroleum and services related to crude oil extraction; excluding surveying    0.0066
Sub-Bituminous Coal                                                                  0.0034
Coking Coal                                                                          0.0033
Basic iron and steel and of ferro-alloys and first products thereof                  0.0028 

top upstream contributors to 'Natural gas and services related to natural gas extraction; excluding surveying':
Item
Natural gas and services related to natural gas extraction; excluding surveying      0.9317

0

#### Step 2 — import share of total use

For each commodity, the share of total use (`U` intermediate + `Yc` final demand) that is
sourced from a **different region** than the buyer. High import share → the trade split is
worth updating from real bilateral statistics (WP3).

In [5]:
U = db.query('U'); Yc = db.query('Yc')
def domestic_and_total(M):
    by_reg = M.T.groupby(M.columns.get_level_values('Region')).sum().T   # rows x region
    reg_pos = {r: i for i, r in enumerate(by_reg.columns)}
    row_region = M.index.get_level_values('Region')
    dom = pd.Series(by_reg.values[np.arange(len(M)), [reg_pos[r] for r in row_region]], index=M.index)
    return dom, M.sum(axis=1)
domU, totU = domestic_and_total(U)
domY, totY = domestic_and_total(Yc)
dom = domU.add(domY, fill_value=0.0); tot = totU.add(totY, fill_value=0.0)
item = tot.index.get_level_values('Item')
import_share = (1 - dom.groupby(item).sum() / tot.groupby(item).sum().replace(0, np.nan)).clip(lower=0)
del U, Yc; gc.collect()
print('highest import shares among carriers:')
print(import_share.reindex([c for c in CARRIER_LABELS if c in import_share.index])
      .sort_values(ascending=False).round(3).head(12).to_string())

highest import shares among carriers:
Item
Anthracite                                                                           0.809
Heavy Fuel Oil                                                                       0.629
Crude petroleum and services related to crude oil extraction; excluding surveying    0.566
Kerosene Type Jet Fuel                                                               0.537
Additives/Blending Components                                                        0.452
Other Hydrocarbons                                                                   0.374
Coking Coal                                                                          0.324
Natural gas and services related to natural gas extraction; excluding surveying      0.301
Coke Oven Coke                                                                       0.279
Aviation Gasoline                                                                    0.278
Gas/Diesel Oil                                 

#### Step 3 — multi-producer structure

From the supply-share matrix `s`: per commodity, the median (across regions) count of
domestic activities holding ≥5% of that commodity's supply — reported as a metric.
The supply-mix **trigger**, however, is the genuine virgin/recycled split: a commodity
qualifies when ≥2% of its supply comes from a `Re-processing of secondary … into new …`
activity (`secondary_share`, computed from `S` in Step 1). This catches steel, aluminium,
copper, glass and plastics uniformly (regardless of how concentrated the recycled share is
across regions) while **not** flagging a bare co-production `n=2` that is not a real mix
(e.g. cement/lime/plaster). Electricity's technology mix is handled separately (done, EMBER).

In [6]:
s2 = db.query('s')
def multiproducer(s2):
    counts = {c: [] for c in commodities}; hhi = {c: [] for c in commodities}
    act_region = s2.index.get_level_values('Region')
    for r in regions:
        blk = s2.loc[act_region == r]
        cols_r = [c for c in s2.columns if c[0] == r]
        vals = blk[cols_r].values
        colsum = vals.sum(axis=0)
        with np.errstate(invalid='ignore', divide='ignore'):
            shares = np.where(colsum > 0, vals / colsum, 0.0)
        n_above = (shares >= SUPPLIER_SHARE).sum(axis=0)
        h = (shares ** 2).sum(axis=0)
        for j, col in enumerate(cols_r):
            counts[col[2]].append(int(n_above[j])); hhi[col[2]].append(float(h[j]))
    n_med = pd.Series({c: (int(np.median(v)) if v else 0) for c, v in counts.items()})
    hhi_min = pd.Series({c: (min(v) if v else 1.0) for c, v in hhi.items()})
    return n_med, hhi_min
n_suppliers, hhi_min = multiproducer(s2)
del s2; gc.collect()
print('commodities with median >=2 domestic producers (>=5% share):')
print(n_suppliers[n_suppliers >= 2].sort_values(ascending=False).to_string())

commodities with median >=2 domestic producers (>=5% share):
Electricity                                                                                             5
Basic iron and steel and of ferro-alloys and first products thereof                                     2
Cattle                                                                                                  2
Cement; lime and plaster                                                                                2
Glass and glass products                                                                                2
N-fertiliser                                                                                            2
P- and other fertiliser                                                                                 2
Plastics; basic                                                                                         2
Pulp                                                                                       

#### Classification → `support/wp1_perimeter.csv`

Rules: **P1** = curated energy carriers; **P2** = curated transition-material shortlist;
**P3** = the rest. `trade` = *pooled* (electricity; gas is a candidate) / *isard* (traded,
import_share ≥ 5%) / *skip*. `supply_mix` = *source* (multi-producer with a WP4 statistic) /
*candidate* (multi-producer, no source yet) / *skip*. Electricity is marked **done** (v3.0
already pools its trade and applies the EMBER mix).

In [7]:
rows = []
for c in commodities:
    is_carrier = c in CARRIER_LABELS
    is_good = unit_of.get(c) in GOODS_UNITS
    imp = float(import_share.get(c, 0.0))
    nsup = int(n_suppliers.get(c, 0))
    contrib = float(energychain_contrib.get(c, 0.0))
    secshare = float(secondary_share.get(c, 0.0))
    prio = 'P1' if is_carrier else ('P2' if c in P2_SHORTLIST else 'P3')
    review = (prio == 'P3') and is_good and contrib >= CONTRIB_MIN
    trade = ('pooled' if c in POOLED else 'isard') if (prio in ('P1', 'P2') and imp >= IMPORT_MIN) else 'skip'
    # supply-mix requires a genuine disaggregated-activity split: the virgin/recycled split on
    # the primary good (electricity's technology mix is handled by the override below). A bare
    # n>=2 without a recycled split (e.g. cement/lime/plaster co-production) is NOT a real mix.
    multi = secshare >= SECONDARY_MIN and is_good
    if prio in ('P1', 'P2') and multi:
        supply_mix = 'source' if c in SUPPLY_MIX_SOURCE else 'candidate'
    else:
        supply_mix = 'skip'
    status = ''
    if c in ('Electricity', 'Electricity need'):
        trade, supply_mix, status = 'pooled', 'source', 'done (v3.0)'
    rows.append({
        'commodity': c, 'unit': unit_of.get(c, ''),
        'priority': prio, 'trade': trade, 'supply_mix': supply_mix, 'status': status,
        'review_candidate': review, 'is_energy_carrier': is_carrier,
        'ghg_intensity_tco2eq_per_unit': round(float(ghg_int.get(c, np.nan)), 4) if pd.notna(ghg_int.get(c, np.nan)) else '',
        'import_share': round(imp, 4), 'n_domestic_producers': nsup,
        'secondary_supply_share': round(secshare, 4),
        'min_domestic_hhi': round(float(hhi_min.get(c, 1.0)), 4),
        'energychain_contrib': round(contrib, 4),
        'supply_mix_source': SUPPLY_MIX_SOURCE.get(c) or P2_SHORTLIST.get(c, ''),
    })

df = pd.DataFrame(rows).sort_values(['priority', 'commodity']).reset_index(drop=True)
df.to_csv('support/wp1_perimeter.csv', index=False)
print('wrote support/wp1_perimeter.csv ({} rows)'.format(len(df)))
print()
print(df.groupby(['priority', 'trade', 'supply_mix']).size().to_string())
print('\nreview candidates (P3 goods contributing to a major energy vector):')
rc = df[df.review_candidate][['commodity', 'energychain_contrib', 'import_share', 'n_domestic_producers']]
print('  none' if rc.empty else rc.to_string(index=False))

wrote support/wp1_perimeter.csv (197 rows)

priority  trade   supply_mix
P1        isard   skip           28
          pooled  source          2
          skip    skip           22
P2        isard   candidate       2
                  skip            7
                  source          3
          skip    skip            3
P3        skip    skip          130

review candidates (P3 goods contributing to a major energy vector):
  none


#### Findings

- **P1 (energy carriers):** ~52 commodities. Traded carriers with import share ≥5%
  (coal qualities, crude, most refined products, natural gas, NGL) are **isard** trade-update
  candidates; electricity is already **pooled + EMBER-mixed** (done, v3.0). Natural gas is the
  natural next pooling candidate.
- **P2 (transition materials):** the shortlist confirms — Basic iron & steel, Aluminium,
  Copper (+ ores + secondary), Cement/clinker, Glass, Plastics. **All five carry an observable
  virgin/recycled supply split** in `S` (a `Re-processing of secondary … into new …` activity
  feeds the primary commodity): steel 17%, aluminium 21%, copper 13%, glass 47%, plastics 5% of
  domestic supply. So steel, aluminium and plastics → supply-mix **source** (worldsteel / IAI /
  OECD); copper and glass → **candidate** (same structure, no mix source listed in WP4 yet —
  ICSG could serve copper). Cement is **excluded**: its `n=2` is cement/lime/plaster
  co-production under one aggregated commodity, not a real virgin/recycled or technology split.
  All these materials trade heavily (import 13–88%) → BACI trade adapters (WP3b).
- **The upstream decomposition promotes no new material into P2** (no P3 good clears the 3%
  contribution bar to a major energy vector). This is expected and worth recording: in this
  hybrid SUT the material embodiment of energy **technologies** lives in capital formation
  (final demand), not in intermediate use, so the P2 materials are included by domain
  knowledge and confirmed here by their own GHG intensity, trade and multi-producer metrics —
  not by the intermediate-footprint decomposition. The decomposition instead cleanly confirms
  the fossil chains (e.g. Gas/Diesel Oil ≈ 50% crude petroleum; Natural gas ≈ 93% own
  extraction).

**Next steps (per the plan):** WP3a energy-carrier trades (IEA/Eurostat) for the isard
carriers, starting gas + coal + crude + refined products; WP4 heat and fuel-blending mixes;
then WP3b BACI material trades + WP4 steel/aluminium/plastics mixes for P2. The committed
`support/wp1_perimeter.csv` is the driver list for those WPs.